# WiDS Global Datathon 2026 — Unified Survival Stack

**Kết hợp:**
- Tri-survival ensemble (GBSA + CoxPH + RSF + Zone-LGB) từ `0-9716-tri-stack`
- Sparse-near subgroup heads (LR + Meta-LR + KNN) + stable-near floors từ `wids-n23`
- False case analysis mới trên OOF predictions

**Pipeline:**
1. Config & Imports
2. Data Loading & Zone Masks
3. Feature Engineering
4. Shared Utility Functions
5. GBSA Ensemble
6. CoxPH Ensemble
7. RSF Ensemble
8. Zone-Split LGB
9. Sparse-Near Subgroup Heads
10. Tri-Stack Zone Blend
11. Sparse/Stable Overlay + Calibration
12. OOF Scoring
13. False Case Analysis
14. Submission Generation

In [ ]:
# ── Step 1: Config & Imports ──────────────────────────────────────────────
import subprocess, sys
def _install(pkg, import_name=None):
    name = import_name or pkg
    try: __import__(name)
    except Exception:
        print(f'[INSTALL] {pkg}')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

_install('scikit-survival', 'sksurv')
_install('lightgbm')
_install('scikit-learn', 'sklearn')

import os, warnings, time as timer
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.isotonic import IsotonicRegression
from scipy import stats as scipy_stats
import lightgbm as lgb
from sksurv.util import Surv
from sksurv.ensemble import GradientBoostingSurvivalAnalysis, RandomSurvivalForest
from sksurv.linear_model import CoxPHSurvivalAnalysis

warnings.filterwarnings('ignore')
np.random.seed(777)

# ── Run mode: 'full' for submission, 'fast' for iteration ─────────────────
RUN_MODE = 'full'

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR = '/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26'
if not os.path.isdir(DATA_DIR):
    DATA_DIR = '../data'
if not os.path.isdir(DATA_DIR):
    DATA_DIR = 'data'
OUTPUT_PATH_A = '/kaggle/working/submission_A.csv' if os.path.isdir('/kaggle/working') else 'submission_A.csv'
OUTPUT_PATH_B = '/kaggle/working/submission_B.csv' if os.path.isdir('/kaggle/working') else 'submission_B.csv'

# ── Zone threshold ─────────────────────────────────────────────────────────
STRAT_THR = 5000   # near < 5km, far >= 5km
HORIZONS_PRED = [12, 24, 48, 72]

# ── GBSA configs (from tri-stack, 10 configs) ─────────────────────────────
GBSA_CONFIGS = [
    {'learning_rate':0.01,  'subsample':0.70, 'max_depth':3, 'min_samples_leaf':12, 'min_samples_split':3, 'n_estimators':1200, 'dropout_rate':0.0},
    {'learning_rate':0.01,  'subsample':0.85, 'max_depth':3, 'min_samples_leaf':15, 'min_samples_split':3, 'n_estimators':1200, 'dropout_rate':0.0},
    {'learning_rate':0.01,  'subsample':0.60, 'max_depth':3, 'min_samples_leaf':12, 'min_samples_split':3, 'n_estimators':1200, 'dropout_rate':0.0},
    {'learning_rate':0.005, 'subsample':0.85, 'max_depth':3, 'min_samples_leaf':12, 'min_samples_split':3, 'n_estimators':2000, 'dropout_rate':0.0},
    {'learning_rate':0.01,  'subsample':0.85, 'max_depth':3, 'min_samples_leaf':20, 'min_samples_split':3, 'n_estimators':1400, 'dropout_rate':0.0},
    {'learning_rate':0.008, 'subsample':0.75, 'max_depth':2, 'min_samples_leaf':15, 'min_samples_split':4, 'n_estimators':1500, 'dropout_rate':0.0},
    {'learning_rate':0.015, 'subsample':0.70, 'max_depth':3, 'min_samples_leaf':10, 'min_samples_split':3, 'n_estimators':1000, 'dropout_rate':0.0},
    {'learning_rate':0.005, 'subsample':0.90, 'max_depth':3, 'min_samples_leaf':18, 'min_samples_split':5, 'n_estimators':2500, 'dropout_rate':0.0},
    {'learning_rate':0.01,  'subsample':0.80, 'max_depth':4, 'min_samples_leaf':12, 'min_samples_split':3, 'n_estimators':1200, 'dropout_rate':0.0},
    {'learning_rate':0.02,  'subsample':0.65, 'max_depth':3, 'min_samples_leaf':10, 'min_samples_split':3, 'n_estimators':800,  'dropout_rate':0.0},
]
GBSA_SEEDS_FULL = (
    123, 456, 789, 777, 666, 1511, 1523, 2025, 2026, 2033,
    279, 239, 70, 77, 31, 2024, 2077, 3077, 123456, 654321,
    4640, 841, 7755, 8525, 2701, 8817, 8864, 4085, 8919, 934,
    4746, 1699, 7401, 7826, 4098, 2921, 1204, 2752, 8384, 1284,
)
GBSA_SEEDS_FAST = tuple(range(42, 52))
GBSA_SEEDS = GBSA_SEEDS_FULL if RUN_MODE == 'full' else GBSA_SEEDS_FAST

# ── CoxPH ──────────────────────────────────────────────────────────────────
COX_ALPHAS = [0.001, 0.01, 0.05, 0.10, 0.50, 1.00, 2.00]
COX_SEEDS_FULL = (123, 456, 789, 777, 666, 1511, 1523, 2025, 2026, 2033,
                   279, 239, 70, 77, 31, 2024, 2077, 3077, 123456, 654321)
COX_SEEDS_FAST = (123, 456, 789)
COX_SEEDS = COX_SEEDS_FULL if RUN_MODE == 'full' else COX_SEEDS_FAST

COX_FEATURES = [
    'dist_km', 'log_distance', 'inv_distance',
    'closing_speed_m_per_h', 'radial_growth_rate_m_per_h',
    'alignment_abs', 'threat_score', 'log_eta', 'eta_effective',
    'area_to_dist_ratio', 'fire_radius_km',
    'num_perimeters_0_5h', 'has_movement',
    'near_speed_rank', 'far_threat_rank',
    'is_summer', 'is_afternoon',
    'zone_near', 'zone_far',
]

# ── RSF ────────────────────────────────────────────────────────────────────
RSF_CONFIGS = [
    {'n_estimators': 200, 'min_samples_leaf': 12, 'max_features': 'sqrt', 'max_depth': None},
    {'n_estimators': 200, 'min_samples_leaf': 18, 'max_features': 'sqrt', 'max_depth': None},
    {'n_estimators': 200, 'min_samples_leaf': 12, 'max_features': 0.5,    'max_depth': 5},
]
RSF_SEEDS_FULL = (123, 456, 789, 777, 666, 1511, 1523, 2025, 2026, 2033,
                   279, 239, 70, 77, 31)
RSF_SEEDS_FAST = (123, 456, 789)
RSF_SEEDS = RSF_SEEDS_FULL if RUN_MODE == 'full' else RSF_SEEDS_FAST

# ── Zone-Split LGB ─────────────────────────────────────────────────────────
NEAR_LGB_FEATURES = [
    'closing_speed_m_per_h', 'radial_growth_rate_m_per_h',
    'alignment_abs', 'num_perimeters_0_5h', 'area_growth_rate_ha_per_h',
    'eta_effective', 'log_eta', 'dist_km', 'threat_score',
    'near_speed_rank', 'event_start_hour', 'is_afternoon', 'fire_urgency',
    'area_first_ha', 'fire_radius_km',
]
FAR_LGB_FEATURES = [
    'dist_km', 'log_distance', 'inv_distance',
    'closing_speed_m_per_h', 'alignment_abs',
    'threat_score', 'log_eta', 'eta_effective',
    'area_to_dist_ratio', 'num_perimeters_0_5h',
    'far_threat_rank', 'is_summer', 'zone_far',
]
NEAR_LGB_CFGS = {
    24: {'max_depth':2, 'learning_rate':0.04, 'n_estimators':200,
         'subsample':0.8, 'colsample_bytree':0.8, 'min_child_samples':4,
         'reg_alpha':0.5, 'reg_lambda':1.5, 'num_leaves':4},
    48: {'max_depth':2, 'learning_rate':0.05, 'n_estimators':150,
         'subsample':0.8, 'colsample_bytree':0.8, 'min_child_samples':3,
         'reg_alpha':0.3, 'reg_lambda':1.0, 'num_leaves':4},
}
FAR_LGB_CFGS = {
    24: {'max_depth':2, 'learning_rate':0.03, 'n_estimators':200,
         'subsample':0.7, 'colsample_bytree':0.7, 'min_child_samples':8,
         'reg_alpha':1.0, 'reg_lambda':3.0, 'num_leaves':4},
    48: {'max_depth':2, 'learning_rate':0.05, 'n_estimators':150,
         'subsample':0.8, 'colsample_bytree':0.8, 'min_child_samples':6,
         'reg_alpha':0.5, 'reg_lambda':2.0, 'num_leaves':4},
}
LGB_NEAR_SEEDS_FULL = (123, 456, 789, 777, 666, 1511, 1523, 2025, 2026, 2033,
                        279, 239, 70, 77, 31, 2024, 2077, 3077, 123456, 654321,
                        2034, 2035, 2036, 1984, 1991, 3255, 1011, 6241, 2790, 6847)
LGB_FAR_SEEDS_FULL  = (8141, 7752, 432, 906, 6217, 7785, 1603, 7609, 965, 2506,
                        3771, 7080, 4963, 7939, 2751, 473, 339, 3675, 5535, 4760,
                        123, 456, 789, 777, 666, 1511, 1523, 2025, 2026, 2033)
LGB_NEAR_SEEDS_FAST = tuple(range(42, 52))
LGB_FAR_SEEDS_FAST  = tuple(range(52, 62))
LGB_NEAR_SEEDS = LGB_NEAR_SEEDS_FULL if RUN_MODE == 'full' else LGB_NEAR_SEEDS_FAST
LGB_FAR_SEEDS  = LGB_FAR_SEEDS_FULL  if RUN_MODE == 'full' else LGB_FAR_SEEDS_FAST

# ── Locked blend weights (validated LB=0.97167) ────────────────────────────
W_GBSA_NEAR_12=0.76; W_COX_NEAR_12=0.12; W_RSF_NEAR_12=0.02; W_LGB_NEAR_12=0.10
W_GBSA_NEAR_24=0.82; W_COX_NEAR_24=0.14; W_RSF_NEAR_24=0.02; W_LGB_NEAR_24=0.02
W_GBSA_NEAR_48=0.73; W_COX_NEAR_48=0.16; W_RSF_NEAR_48=0.03; W_LGB_NEAR_48=0.08
W_GBSA_FAR_24=0.62;  W_COX_FAR_24=0.25;  W_RSF_FAR_24=0.06;  W_LGB_FAR_24=0.07
W_GBSA_FAR_48=0.35;  W_COX_FAR_48=0.22;  W_RSF_FAR_48=0.06;  W_LGB_FAR_48=0.37

# ── Calibration constants ──────────────────────────────────────────────────
POWER_CAL_24     = 1.1
RIGHT_TAIL_SHIFT = 0.09
LEFT_TAIL_CAP    = 0.01
LEFT_TAIL_THR    = 0.03

# ── Stable-near hard floors ────────────────────────────────────────────────
STABLE_FLOORS = {12: 0.90, 24: 0.965, 48: 0.995}

# ── Sparse-near features ───────────────────────────────────────────────────
SPARSE_FEATURES = [
    'dt_first_last_0_5h', 'num_perimeters_0_5h', 'eta_effective',
    'alignment_abs', 'log_area_dist_ratio', 'effective_closing_speed',
    'log_distance', 'fire_urgency', 'inv_distance',
]
SPARSE_META_FEATURES = [
    'dt_first_last_0_5h', 'num_perimeters_0_5h', 'eta_effective',
    'alignment_abs', 'log_area_dist_ratio', 'effective_closing_speed',
    'log_distance', 'inv_distance',
]
KNN_FEATURES = [
    'log_area_dist_ratio', 'dist_min_ci_0_5h', 'event_start_hour',
    'event_start_month', 'inv_distance', 'log_distance',
]

# ── Sparse-near seeds ──────────────────────────────────────────────────────
SPARSE_LR_SEEDS = (123, 456, 789, 777, 666, 1511, 1523, 2025, 2026, 2033)
META_SEEDS      = (101, 202, 303, 404, 505, 606, 707, 808, 909, 1001)
KNN_K_LIST      = [5, 7, 9, 11, 15]

# ── Sparse-near blend weights ──────────────────────────────────────────────
# (from wids-n23, tuned against 2-model stack — use as-is on first run)
SPARSE_W_META = {12: 0.42, 24: 0.30, 48: 0.38}   # meta-LR weight
SPARSE_W_LR   = {12: 0.08, 24: 0.05, 48: 0.05}   # LR stabilizer weight
SPARSE_W_KNN  = {12: 0.10, 24: 0.10, 48: 0.16}   # KNN nudge weight

DO_OOF      = True
CV_BAG_TEST = True

print(f'RUN_MODE={RUN_MODE}')
print(f'GBSA: {len(GBSA_CONFIGS)} configs x {len(GBSA_SEEDS)} seeds')
print(f'CoxPH: {len(COX_ALPHAS)} alphas x {len(COX_SEEDS)} seeds')
print(f'RSF: {len(RSF_CONFIGS)} configs x {len(RSF_SEEDS)} seeds')
print(f'LGB near: {len(LGB_NEAR_SEEDS)} seeds | far: {len(LGB_FAR_SEEDS)} seeds')

In [ ]:
# ── Step 2: Data Loading & Zone Masks ─────────────────────────────────────
train_df   = pd.read_csv(f'{DATA_DIR}/train.csv')
test_df    = pd.read_csv(f'{DATA_DIR}/test.csv')
sample_sub = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

print('Train:', train_df.shape, '| Test:', test_df.shape)
print('Events:', train_df['event'].value_counts().to_dict())

dist_train   = train_df['dist_min_ci_0_5h'].values
dist_test    = test_df['dist_min_ci_0_5h'].values
lowtemp_train = train_df['low_temporal_resolution_0_5h'].astype(int).values
lowtemp_test  = test_df['low_temporal_resolution_0_5h'].astype(int).values
event_values  = train_df['event'].values
time_values   = train_df['time_to_hit_hours'].values

# Zone masks — computed once, reused everywhere
near_train   = dist_train < STRAT_THR
far_train    = ~near_train
stable_train = near_train & (lowtemp_train == 0)
sparse_train = near_train & (lowtemp_train == 1)

near_test    = dist_test < STRAT_THR
far_test     = ~near_test
stable_test  = near_test & (lowtemp_test == 0)
sparse_test  = near_test & (lowtemp_test == 1)

# Empirical hit rates in stable-near (used as floor anchors)
def rate_within(mask, horizon):
    return float((train_df.loc[mask, 'time_to_hit_hours'] <= horizon).mean())

p12_stable = rate_within(stable_train, 12)
p24_stable = rate_within(stable_train, 24)
p48_stable = rate_within(stable_train, 48)

# Survival arrays
y_surv = Surv.from_arrays(event=train_df['event'].astype(bool),
                            time=train_df['time_to_hit_hours'])

print(f'Near (<{STRAT_THR/1000:.0f}km): train={near_train.sum()} test={near_test.sum()}')
print(f'Far (>={STRAT_THR/1000:.0f}km): train={far_train.sum()} test={far_test.sum()}')
print(f'Stable-near: train={stable_train.sum()} test={stable_test.sum()}')
print(f'Sparse-near: train={sparse_train.sum()} test={sparse_test.sum()}')
print(f'p12_stable={p12_stable:.4f} p24_stable={p24_stable:.4f} p48_stable={p48_stable:.4f}')

In [ ]:
# ── Step 3: Feature Engineering ───────────────────────────────────────────
# Always use fit_df=train_df to anchor zone-rank features to training distribution

def create_features(df, fit_df=None):
    ref    = fit_df if fit_df is not None else df
    result = df.copy()
    dist       = result['dist_min_ci_0_5h'].clip(lower=1)
    speed      = result['closing_speed_m_per_h']
    perimeters = result['num_perimeters_0_5h']
    area_first = result['area_first_ha']

    result['log_distance']    = np.log1p(dist)
    result['inv_distance']    = 1 / (dist / 1000 + 0.1)
    result['inv_distance_sq'] = result['inv_distance'] ** 2
    result['sqrt_distance']   = np.sqrt(dist)
    result['dist_km']         = dist / 1000
    result['dist_km_sq']      = (dist / 1000) ** 2
    result['dist_km_cb']      = (dist / 1000) ** 3
    result['dist_rank']       = dist.rank(pct=True)

    fire_radius              = np.sqrt(area_first * 10000 / np.pi)
    result['fire_radius_km'] = fire_radius / 1000
    result['radius_to_dist'] = fire_radius / dist
    result['area_to_dist_ratio']  = area_first / (dist / 1000 + 0.1)
    result['log_area_dist_ratio'] = np.log1p(area_first) - np.log1p(dist)

    result['has_movement']  = (perimeters > 1).astype(float)
    closing_pos             = speed.clip(lower=0)
    result['eta_hours']     = np.where(closing_pos > 0.01, dist / closing_pos, 9999).clip(max=9999)
    result['log_eta']       = np.log1p(result['eta_hours'].clip(0, 9999))
    radial_growth           = result['radial_growth_rate_m_per_h'].clip(lower=0)
    effective_closing       = closing_pos + radial_growth
    result['effective_closing_speed'] = effective_closing
    result['eta_effective'] = np.where(effective_closing > 0.01, dist / effective_closing, 9999).clip(max=9999)
    result['threat_score']  = result['alignment_abs'] * speed / np.log1p(dist)
    result['threat_score_sq'] = result['threat_score'] ** 2
    result['fire_urgency']  = perimeters * speed
    result['growth_intensity'] = result['area_growth_rate_ha_per_h'] * perimeters

    result['zone_near']    = (dist < 5000).astype(float)
    result['zone_warning'] = ((dist >= 5000) & (dist < 10000)).astype(float)
    result['zone_far']     = (dist >= 10000).astype(float)

    # Zone-rank features anchored to reference distribution (fit_df)
    ref_near_mask = ref['dist_min_ci_0_5h'].clip(lower=1) < 5000
    ref_far_mask  = ~ref_near_mask
    near_speed_ref = ref.loc[ref_near_mask, 'closing_speed_m_per_h'].values
    far_threat_ref = (
        ref.loc[ref_far_mask, 'alignment_abs'] *
        ref.loc[ref_far_mask, 'closing_speed_m_per_h'] /
        np.log1p(ref.loc[ref_far_mask, 'dist_min_ci_0_5h'].clip(lower=1))
    ).values

    def rank_against_ref(vals, ref_vals):
        return np.array([(ref_vals < v).mean() for v in vals])

    cur_near_mask = dist < 5000
    cur_far_mask  = ~cur_near_mask
    near_speed_rank = np.zeros(len(result))
    far_threat_rank = np.zeros(len(result))
    if cur_near_mask.sum() > 0:
        near_speed_rank[cur_near_mask.values] = rank_against_ref(
            speed[cur_near_mask].values, near_speed_ref)
    if cur_far_mask.sum() > 0:
        threat_cur_far = (
            result.loc[cur_far_mask, 'alignment_abs'] *
            speed[cur_far_mask] /
            np.log1p(dist[cur_far_mask])
        ).values
        far_threat_rank[cur_far_mask.values] = rank_against_ref(
            threat_cur_far, far_threat_ref)
    result['near_speed_rank'] = near_speed_rank
    result['far_threat_rank'] = far_threat_rank

    result['is_summer']    = result['event_start_month'].isin([6, 7, 8]).astype(float)
    result['is_afternoon'] = ((result['event_start_hour'] >= 12) &
                               (result['event_start_hour'] < 20)).astype(float)

    drop_cols = [
        'relative_growth_0_5h', 'projected_advance_m',
        'centroid_displacement_m', 'centroid_speed_m_per_h',
        'closing_speed_abs_m_per_h', 'area_growth_abs_0_5h',
    ]
    result = result.drop(columns=[c for c in drop_cols if c in result.columns])
    result = result.replace([np.inf, -np.inf], np.nan).fillna(0)
    return result

train_processed = create_features(train_df, fit_df=train_df)
test_processed  = create_features(test_df,  fit_df=train_df)

n_eng = len([c for c in train_processed.columns if c not in ['event_id','event','time_to_hit_hours']])
print(f'Engineered features: {n_eng}')

In [ ]:
# ── Step 4: Shared Utility Functions ──────────────────────────────────────

def get_surv_predictions(model, X):
    surv_fns = model.predict_survival_function(X)
    preds = np.empty((len(surv_fns), len(HORIZONS_PRED)), dtype=float)
    for i, fn in enumerate(surv_fns):
        t_min, t_max = fn.domain
        preds[i, :] = fn(np.clip(HORIZONS_PRED, t_min, t_max))
    return 1.0 - preds

def make_binary_target(time_vals, event_vals, horizon):
    unknown = (event_vals == 0) & (time_vals < horizon)
    y = ((event_vals == 1) & (time_vals <= horizon)).astype(float)
    return y, ~unknown

def compute_ipcw_weights(times, events, horizon):
    unique_t = np.sort(np.unique(times))
    surv = np.ones(len(unique_t))
    for i, t in enumerate(unique_t):
        at_risk = (times >= t).sum()
        cens    = ((times == t) & (events == 0)).sum()
        if at_risk > 0: surv[i] = 1 - cens / at_risk
        if i > 0: surv[i] *= surv[i - 1]
    def G(t):
        idx = np.searchsorted(unique_t, t, side='right') - 1
        return max(surv[idx], 0.01) if idx >= 0 else 1.0
    weights = np.ones(len(times))
    for i in range(len(times)):
        if events[i] == 1 and times[i] <= horizon:   weights[i] = 1.0 / G(times[i])
        elif times[i] >= horizon:                     weights[i] = 1.0 / G(horizon)
    return weights

def enforce_monotonicity(preds):
    result = np.clip(preds, 0, 1)
    for i in range(1, result.shape[1]):
        result[:, i] = np.maximum(result[:, i], result[:, i - 1])
    return result

def compute_c_index(time, event, risk):
    n = len(time); concordant = comparable = 0
    for i in range(n):
        if event[i] != 1: continue
        for j in range(n):
            if i == j or time[i] >= time[j]: continue
            comparable += 1
            if risk[i] > risk[j]:   concordant += 1
            elif risk[i] == risk[j]: concordant += 0.5
    return concordant / comparable if comparable > 0 else 0.5

def compute_brier(time, event, prob, horizon):
    valid  = ~((event == 0) & (time < horizon))
    if valid.sum() == 0: return 0.25
    y_true = ((event == 1) & (time <= horizon)).astype(float)[valid]
    return float(np.mean((np.clip(prob[valid], 0, 1) - y_true) ** 2))

def compute_hybrid_score(time, event, p24, p48, p72):
    risk  = 0.3 * p24 + 0.4 * p48 + 0.3 * p72
    c_idx = compute_c_index(time, event, risk)
    b24   = compute_brier(time, event, p24, 24)
    b48   = compute_brier(time, event, p48, 48)
    b72   = compute_brier(time, event, p72, 72)
    wb    = 0.3 * b24 + 0.4 * b48 + 0.3 * b72
    return 0.3 * c_idx + 0.7 * (1 - wb), c_idx, wb

def right_tail_shift(preds, shift=RIGHT_TAIL_SHIFT, threshold=0.9):
    return np.where(preds >= threshold, preds + shift, preds).clip(0, 1)

def left_tail_push(preds, far_mask, threshold=LEFT_TAIL_THR, cap=LEFT_TAIL_CAP):
    result = preds.copy()
    for col in range(preds.shape[1] - 1):
        row_mask = far_mask & (preds[:, col] < threshold)
        result[row_mask, col] = cap
    return result

def fit_isotonic_far_zone(oof_preds, time_vals, event_vals, far_mask, horizon, col_idx):
    mask_valid = ~((event_vals == 0) & (time_vals < horizon))
    y_true     = ((event_vals == 1) & (time_vals <= horizon)).astype(float)
    fit_mask   = far_mask & mask_valid
    X_iso = oof_preds[fit_mask, col_idx]
    y_iso = y_true[fit_mask]
    iso   = IsotonicRegression(out_of_bounds='clip', increasing=True)
    iso.fit(X_iso, y_iso)
    b_before = compute_brier(time_vals[far_mask], event_vals[far_mask], oof_preds[far_mask, col_idx], horizon)
    oof_cal  = iso.predict(oof_preds[far_mask, col_idx])
    b_after  = compute_brier(time_vals[far_mask], event_vals[far_mask], oof_cal, horizon)
    gain     = b_before - b_after
    print(f'  Isotonic {horizon}h far-zone OOF: B={b_before:.5f} -> {b_after:.5f} (gain={gain:+.5f}{"  <<NOISE" if gain < 0.0003 else ""})')
    return iso, gain

print('Utility functions defined.')

In [ ]:
# ── Step 5: GBSA Ensemble ──────────────────────────────────────────────────
X_gbsa_train = train_df.drop(columns=['event_id', 'event', 'time_to_hit_hours'])
X_gbsa_test  = test_df.drop(columns=['event_id'])

oof_gbsa  = np.zeros((len(X_gbsa_train), 4))
test_gbsa = np.zeros((len(X_gbsa_test), 4))

print(f'GBSA: {len(GBSA_CONFIGS)} configs x {len(GBSA_SEEDS)} seeds x 5-fold CV-bag')
t0 = timer.time()

for cfg_idx, cfg in enumerate(GBSA_CONFIGS, 1):
    cfg_oof  = np.zeros_like(oof_gbsa)
    cfg_test = np.zeros_like(test_gbsa)
    for seed in GBSA_SEEDS:
        seed_oof  = np.zeros_like(oof_gbsa)
        seed_test = np.zeros_like(test_gbsa)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        for tr_idx, va_idx in cv.split(X_gbsa_train, event_values):
            m = GradientBoostingSurvivalAnalysis(**{**cfg, 'random_state': seed})
            m.fit(X_gbsa_train.iloc[tr_idx], y_surv[tr_idx])
            seed_oof[va_idx] = get_surv_predictions(m, X_gbsa_train.iloc[va_idx])
            if CV_BAG_TEST:
                seed_test += get_surv_predictions(m, X_gbsa_test) / 5
        cfg_oof  += seed_oof  / len(GBSA_SEEDS)
        cfg_test += seed_test / len(GBSA_SEEDS)
    oof_gbsa  += cfg_oof  / len(GBSA_CONFIGS)
    test_gbsa += cfg_test / len(GBSA_CONFIGS)
    elapsed = timer.time() - t0
    print(f'  cfg {cfg_idx}/{len(GBSA_CONFIGS)} done [{elapsed/60:.1f}m]')

# Save raw copies before power calibration
oof_gbsa_raw  = oof_gbsa.copy()
test_gbsa_raw = test_gbsa.copy()

# Power calibration on 24h
oof_gbsa[:,1]  = np.clip(oof_gbsa[:,1]  ** POWER_CAL_24, 0, 1)
test_gbsa[:,1] = np.clip(test_gbsa[:,1] ** POWER_CAL_24, 0, 1)

print(f'GBSA done. [{(timer.time()-t0)/60:.1f}m total]')

In [ ]:
# ── Step 6: CoxPH Ensemble ────────────────────────────────────────────────
avail_cox  = [c for c in COX_FEATURES if c in train_processed.columns]
X_cox_train = train_processed[avail_cox].copy()
X_cox_test  = test_processed[[c for c in avail_cox if c in test_processed.columns]].copy()

scaler_cox = StandardScaler()
X_cox_train_sc = pd.DataFrame(scaler_cox.fit_transform(X_cox_train),
                                columns=X_cox_train.columns, index=X_cox_train.index)
X_cox_test_sc  = pd.DataFrame(scaler_cox.transform(X_cox_test),
                                columns=X_cox_test.columns, index=X_cox_test.index)

oof_cox  = np.zeros((len(X_cox_train_sc), 4))
test_cox = np.zeros((len(X_cox_test_sc), 4))

print(f'CoxPH: {len(COX_ALPHAS)} alphas x {len(COX_SEEDS)} seeds x 5-fold CV-bag')
t0 = timer.time()

for alpha_idx, alpha in enumerate(COX_ALPHAS, 1):
    alpha_oof  = np.zeros_like(oof_cox)
    alpha_test = np.zeros_like(test_cox)
    for seed in COX_SEEDS:
        seed_oof  = np.zeros_like(oof_cox)
        seed_test = np.zeros_like(test_cox)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        for tr_idx, va_idx in cv.split(X_cox_train_sc, event_values):
            m = CoxPHSurvivalAnalysis(alpha=alpha)
            try:
                m.fit(X_cox_train_sc.iloc[tr_idx], y_surv[tr_idx])
                seed_oof[va_idx] = get_surv_predictions(m, X_cox_train_sc.iloc[va_idx])
                if CV_BAG_TEST:
                    seed_test += get_surv_predictions(m, X_cox_test_sc) / 5
            except Exception:
                seed_oof[va_idx] = 0.5
                if CV_BAG_TEST: seed_test += 0.5 / 5
        alpha_oof  += seed_oof  / len(COX_SEEDS)
        alpha_test += seed_test / len(COX_SEEDS)
    oof_cox  += alpha_oof  / len(COX_ALPHAS)
    test_cox += alpha_test / len(COX_ALPHAS)
    print(f'  alpha={alpha} done [{(timer.time()-t0)/60:.1f}m]')

print('CoxPH done.')

In [ ]:
# ── Step 7: RSF Ensemble ──────────────────────────────────────────────────
X_rsf_train = train_df.drop(columns=['event_id', 'event', 'time_to_hit_hours'])
X_rsf_test  = test_df.drop(columns=['event_id'])

oof_rsf  = np.zeros((len(X_rsf_train), 4))
test_rsf = np.zeros((len(X_rsf_test), 4))

print(f'RSF: {len(RSF_CONFIGS)} configs x {len(RSF_SEEDS)} seeds x 5-fold CV-bag')
t0 = timer.time()

for cfg_idx, cfg in enumerate(RSF_CONFIGS, 1):
    cfg_oof  = np.zeros_like(oof_rsf)
    cfg_test = np.zeros_like(test_rsf)
    for seed in RSF_SEEDS:
        seed_oof  = np.zeros_like(oof_rsf)
        seed_test = np.zeros_like(test_rsf)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        for tr_idx, va_idx in cv.split(X_rsf_train, event_values):
            m = RandomSurvivalForest(**{**cfg, 'random_state': seed, 'n_jobs': -1})
            m.fit(X_rsf_train.iloc[tr_idx], y_surv[tr_idx])
            seed_oof[va_idx] = get_surv_predictions(m, X_rsf_train.iloc[va_idx])
            if CV_BAG_TEST:
                seed_test += get_surv_predictions(m, X_rsf_test) / 5
        cfg_oof  += seed_oof  / len(RSF_SEEDS)
        cfg_test += seed_test / len(RSF_SEEDS)
    oof_rsf  += cfg_oof  / len(RSF_CONFIGS)
    test_rsf += cfg_test / len(RSF_CONFIGS)
    print(f'  cfg {cfg_idx}/{len(RSF_CONFIGS)} done [{(timer.time()-t0)/60:.1f}m]')

print('RSF done.')

In [ ]:
# ── Step 8: Zone-Split LGB (IPCW Calibrators) ────────────────────────────
avail_near = [c for c in NEAR_LGB_FEATURES if c in train_processed.columns]
avail_far  = [c for c in FAR_LGB_FEATURES  if c in train_processed.columns]

X_near_tr = train_processed[avail_near]
X_near_te = test_processed[[c for c in avail_near if c in test_processed.columns]]
X_far_tr  = train_processed[avail_far]
X_far_te  = test_processed[[c for c in avail_far  if c in test_processed.columns]]

lgb_near_oof, lgb_near_test = {}, {}
lgb_far_oof,  lgb_far_test  = {}, {}

print(f'Zone-Split LGB: {len(LGB_NEAR_SEEDS)} near seeds | {len(LGB_FAR_SEEDS)} far seeds')
t0 = timer.time()

for horizon in [24, 48]:
    y_bin, mask = make_binary_target(time_values, event_values, horizon)
    valid_idx   = np.where(mask)[0]

    for zone, cfg_d, X_tr, X_te, seeds, oof_d, test_d in [
        ('near', NEAR_LGB_CFGS, X_near_tr, X_near_te, LGB_NEAR_SEEDS, lgb_near_oof, lgb_near_test),
        ('far',  FAR_LGB_CFGS,  X_far_tr,  X_far_te,  LGB_FAR_SEEDS,  lgb_far_oof,  lgb_far_test),
    ]:
        cfg      = cfg_d[horizon]
        all_oof  = np.zeros(len(X_tr))
        all_test = np.zeros(len(X_te))

        for seed in seeds:
            seed_oof = np.zeros(len(X_tr))
            last_m   = None
            cv       = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
            for tr_v, va_v in cv.split(valid_idx, y_bin[mask]):
                tr_idx, va_idx = valid_idx[tr_v], valid_idx[va_v]
                ipcw_w = compute_ipcw_weights(time_values[tr_idx], event_values[tr_idx], horizon)
                m = lgb.LGBMClassifier(**cfg, objective='binary', random_state=seed, verbose=-1)
                m.fit(X_tr.iloc[tr_idx], y_bin[tr_idx], sample_weight=ipcw_w)
                seed_oof[va_idx] = m.predict_proba(X_tr.iloc[va_idx])[:, 1]
                last_m = m
            # Gap-fill censored samples outside valid_mask
            cens_idx = np.where(~mask)[0]
            if len(cens_idx) > 0 and last_m is not None:
                seed_oof[cens_idx] = last_m.predict_proba(X_tr.iloc[cens_idx])[:, 1]
            all_oof += seed_oof
            ipcw_full = compute_ipcw_weights(time_values[valid_idx], event_values[valid_idx], horizon)
            mf = lgb.LGBMClassifier(**cfg, objective='binary', random_state=seed, verbose=-1)
            mf.fit(X_tr.iloc[valid_idx], y_bin[valid_idx], sample_weight=ipcw_full)
            all_test += mf.predict_proba(X_te)[:, 1]

        oof_d[horizon]  = all_oof  / len(seeds)
        test_d[horizon] = all_test / len(seeds)

    b_near = compute_brier(time_values, event_values, np.clip(lgb_near_oof[horizon], 0, 1), horizon)
    b_far  = compute_brier(time_values, event_values, np.clip(lgb_far_oof[horizon],  0, 1), horizon)
    print(f'  {horizon}h -> Near-LGB B={b_near:.5f} | Far-LGB B={b_far:.5f} [{(timer.time()-t0)/60:.1f}m]')

print('Zone-split LGB done.')

In [ ]:
# ── Step 9: Sparse-Near Subgroup Heads ────────────────────────────────────

# Subsets for sparse-near
X_sparse_tr = train_processed.loc[sparse_train, SPARSE_FEATURES].reset_index(drop=True)
X_sparse_te = test_processed.loc[sparse_test,  SPARSE_FEATURES].reset_index(drop=True)
time_sparse  = train_df.loc[sparse_train, 'time_to_hit_hours'].values

# ── 9a. Sparse-LR Heads ────────────────────────────────────────────────────
sparse_lr_oof  = {}
sparse_lr_test = {}

for horizon, C in [(12, 0.70), (24, 0.40), (48, 0.25)]:
    y_h    = (time_sparse <= horizon).astype(int)
    counts = np.bincount(y_h, minlength=2)
    n_splits = max(2, min(5, int(counts.min())))

    oof_sum = np.zeros(len(X_sparse_tr))
    oof_cnt = np.zeros(len(X_sparse_tr))
    test_sum = np.zeros(len(X_sparse_te))

    for seed in SPARSE_LR_SEEDS:
        seed_test = np.zeros(len(X_sparse_te))
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_idx, va_idx in cv.split(X_sparse_tr, y_h):
            m = Pipeline([
                ('sc', StandardScaler()),
                ('lr', LogisticRegression(C=C, max_iter=5000, solver='lbfgs', class_weight='balanced'))
            ])
            m.fit(X_sparse_tr.iloc[tr_idx], y_h[tr_idx])
            oof_sum[va_idx] += m.predict_proba(X_sparse_tr.iloc[va_idx])[:, 1]
            oof_cnt[va_idx] += 1
            seed_test += m.predict_proba(X_sparse_te)[:, 1] / n_splits
        test_sum += seed_test / len(SPARSE_LR_SEEDS)

    oof = np.divide(oof_sum, np.maximum(oof_cnt, 1))
    oof[oof_cnt == 0] = float(y_h.mean())
    sparse_lr_oof[horizon]  = np.clip(oof, 0, 1)
    sparse_lr_test[horizon] = np.clip(test_sum, 0, 1)
    print(f'Sparse-LR {horizon}h | n={len(y_h)} pos={int(y_h.sum())} folds={n_splits}')

# ── 9b. Sparse Meta-LR ────────────────────────────────────────────────────
X_sparse_meta_tr = train_processed.loc[sparse_train, SPARSE_META_FEATURES].reset_index(drop=True)
X_sparse_meta_te = test_processed.loc[sparse_test,  SPARSE_META_FEATURES].reset_index(drop=True)

sparse_meta_oof  = {}
sparse_meta_test = {}

for horizon, C in [(12, 0.55), (24, 0.32), (48, 0.22)]:
    y_h    = (time_sparse <= horizon).astype(int)
    counts = np.bincount(y_h, minlength=2)
    n_splits = max(2, min(5, int(counts.min())))

    # Stack: [LR_oof] + [lgb_near_oof for h=24,48] + raw meta features
    base_tr = [sparse_lr_oof[horizon]]
    base_te = [sparse_lr_test[horizon]]
    if horizon in (24, 48):
        base_tr.append(lgb_near_oof[horizon][sparse_train])
        base_te.append(lgb_near_test[horizon][sparse_test])

    M_tr = np.concatenate([np.column_stack(base_tr), X_sparse_meta_tr.values], axis=1)
    M_te = np.concatenate([np.column_stack(base_te), X_sparse_meta_te.values], axis=1)

    test_sum = np.zeros(len(M_te))
    for seed in META_SEEDS:
        seed_test = np.zeros(len(M_te))
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_idx, va_idx in cv.split(M_tr, y_h):
            m = Pipeline([
                ('sc', StandardScaler()),
                ('lr', LogisticRegression(C=C, max_iter=5000, solver='lbfgs', class_weight='balanced'))
            ])
            m.fit(M_tr[tr_idx], y_h[tr_idx])
            seed_test += m.predict_proba(M_te)[:, 1] / n_splits
        test_sum += seed_test / len(META_SEEDS)

    # OOF for sparse-train (for error analysis)
    oof_sum = np.zeros(len(M_tr))
    oof_cnt = np.zeros(len(M_tr))
    for seed in META_SEEDS:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_idx, va_idx in cv.split(M_tr, y_h):
            m = Pipeline([
                ('sc', StandardScaler()),
                ('lr', LogisticRegression(C=C, max_iter=5000, solver='lbfgs', class_weight='balanced'))
            ])
            m.fit(M_tr[tr_idx], y_h[tr_idx])
            oof_sum[va_idx] += m.predict_proba(M_tr[va_idx])[:, 1]
            oof_cnt[va_idx] += 1
    oof = np.divide(oof_sum, np.maximum(oof_cnt, 1))
    oof[oof_cnt == 0] = float(y_h.mean())

    sparse_meta_oof[horizon]  = np.clip(oof, 0, 1)
    sparse_meta_test[horizon] = np.clip(test_sum, 0, 1)
    print(f'Sparse Meta-LR {horizon}h done | folds={n_splits}')

# ── 9c. Sparse KNN Analog Timing ──────────────────────────────────────────
X_knn_tr = train_processed.loc[sparse_train, KNN_FEATURES].reset_index(drop=True)
X_knn_te = test_processed.loc[sparse_test,  KNN_FEATURES].reset_index(drop=True)
time_knn = time_sparse

sparse_knn_test = {12: np.zeros(len(X_knn_te)), 24: np.zeros(len(X_knn_te)), 48: np.zeros(len(X_knn_te))}

if len(X_knn_te) > 0:
    scaler_knn = StandardScaler()
    Z_tr = scaler_knn.fit_transform(X_knn_tr)
    Z_te = scaler_knn.transform(X_knn_te)

    k_list = [k for k in KNN_K_LIST if k <= len(Z_tr)]
    for k in k_list:
        nn = NearestNeighbors(n_neighbors=k, metric='euclidean')
        nn.fit(Z_tr)
        dists, inds = nn.kneighbors(Z_te)
        weights = 1.0 / (dists + 0.15)
        weights = weights / weights.sum(axis=1, keepdims=True)
        for h in [12, 24, 48]:
            y_h = (time_knn <= h).astype(float)
            sparse_knn_test[h] += (weights * y_h[inds]).sum(axis=1) / len(k_list)

for h in [12, 24, 48]:
    sparse_knn_test[h] = np.clip(sparse_knn_test[h], 0, 1)
    print(f'KNN {h}h mean={sparse_knn_test[h].mean() if len(X_knn_te) > 0 else 0:.4f}')

In [ ]:
# ── Step 10: Tri-Stack Zone Blend ─────────────────────────────────────────

def zone_blend_h(gbsa, cox, rsf, lgb_near, lgb_far, near_mask,
                  wg_n, wc_n, wr_n, wl_n, wg_f, wc_f, wr_f, wl_f):
    near = wg_n*gbsa + wc_n*cox + wr_n*rsf + wl_n*lgb_near
    far  = wg_f*gbsa + wc_f*cox + wr_f*rsf + wl_f*lgb_far
    return np.where(near_mask, near, far)

oof_blend  = np.zeros((len(train_df), 4))
test_blend = np.zeros((len(test_df), 4))

# 12h — no far-zone LGB model, far uses GBSA only
oof_blend[:,0] = zone_blend_h(
    oof_gbsa[:,0], oof_cox[:,0], oof_rsf[:,0],
    lgb_near_oof[24], lgb_far_oof[24], near_train,
    W_GBSA_NEAR_12, W_COX_NEAR_12, W_RSF_NEAR_12, W_LGB_NEAR_12,
    1.0, 0.0, 0.0, 0.0)
test_blend[:,0] = zone_blend_h(
    test_gbsa[:,0], test_cox[:,0], test_rsf[:,0],
    lgb_near_test[24], lgb_far_test[24], near_test,
    W_GBSA_NEAR_12, W_COX_NEAR_12, W_RSF_NEAR_12, W_LGB_NEAR_12,
    1.0, 0.0, 0.0, 0.0)

# 24h
oof_blend[:,1] = zone_blend_h(
    oof_gbsa[:,1], oof_cox[:,1], oof_rsf[:,1],
    lgb_near_oof[24], lgb_far_oof[24], near_train,
    W_GBSA_NEAR_24, W_COX_NEAR_24, W_RSF_NEAR_24, W_LGB_NEAR_24,
    W_GBSA_FAR_24,  W_COX_FAR_24,  W_RSF_FAR_24,  W_LGB_FAR_24)
test_blend[:,1] = zone_blend_h(
    test_gbsa[:,1], test_cox[:,1], test_rsf[:,1],
    lgb_near_test[24], lgb_far_test[24], near_test,
    W_GBSA_NEAR_24, W_COX_NEAR_24, W_RSF_NEAR_24, W_LGB_NEAR_24,
    W_GBSA_FAR_24,  W_COX_FAR_24,  W_RSF_FAR_24,  W_LGB_FAR_24)

# 48h
oof_blend[:,2] = zone_blend_h(
    oof_gbsa[:,2], oof_cox[:,2], oof_rsf[:,2],
    lgb_near_oof[48], lgb_far_oof[48], near_train,
    W_GBSA_NEAR_48, W_COX_NEAR_48, W_RSF_NEAR_48, W_LGB_NEAR_48,
    W_GBSA_FAR_48,  W_COX_FAR_48,  W_RSF_FAR_48,  W_LGB_FAR_48)
test_blend[:,2] = zone_blend_h(
    test_gbsa[:,2], test_cox[:,2], test_rsf[:,2],
    lgb_near_test[48], lgb_far_test[48], near_test,
    W_GBSA_NEAR_48, W_COX_NEAR_48, W_RSF_NEAR_48, W_LGB_NEAR_48,
    W_GBSA_FAR_48,  W_COX_FAR_48,  W_RSF_FAR_48,  W_LGB_FAR_48)

# 72h — deterministic
oof_blend[:,3]  = 1.0
test_blend[:,3] = 1.0

print('Tri-stack zone blend done.')
hybrid_raw, c_raw, wb_raw = compute_hybrid_score(
    time_values, event_values, oof_blend[:,1], oof_blend[:,2], oof_blend[:,3])
print(f'Pre-calibration OOF Hybrid: {hybrid_raw:.5f}  C-Index: {c_raw:.4f}  WBrier: {wb_raw:.5f}')

In [ ]:
# ── Step 11: Sparse/Stable Overlay + Calibration ──────────────────────────
# CRITICAL: follow this exact order

def apply_sparse_overlay(blend, sparse_mask, meta, lr, knn=None):
    """Apply sparse-near subgroup heads. knn=None for OOF (no KNN OOF)."""
    for h_idx, h in enumerate([12, 24, 48]):
        w_meta = SPARSE_W_META[h]
        w_lr   = SPARSE_W_LR[h]
        # Layer 1: tri-stack + meta-LR
        blend[sparse_mask, h_idx] = (
            (1 - w_meta) * blend[sparse_mask, h_idx] + w_meta * meta[h]
        )
        # Layer 2: LR stabilizer
        blend[sparse_mask, h_idx] = (
            (1 - w_lr) * blend[sparse_mask, h_idx] + w_lr * lr[h]
        )
        # Layer 3: KNN nudge (test only)
        if knn is not None:
            w_knn = SPARSE_W_KNN[h]
            blend[sparse_mask, h_idx] = (
                (1 - w_knn) * blend[sparse_mask, h_idx] + w_knn * knn[h]
            )
    # Anti-overfit floor for 48h sparse
    blend[sparse_mask, 2] = np.maximum(blend[sparse_mask, 2], 0.88)
    return blend

# 11a. Sparse-near overlay
# OOF: use meta_oof and lr_oof (no KNN OOF)
apply_sparse_overlay(oof_blend, sparse_train, sparse_meta_oof, sparse_lr_oof, knn=None)
# Test: use meta_test, lr_test, and knn_test
apply_sparse_overlay(test_blend, sparse_test, sparse_meta_test, sparse_lr_test, knn=sparse_knn_test)

# 11b. Stable-near hard floors
for blend, stable_mask in [(oof_blend, stable_train), (test_blend, stable_test)]:
    blend[stable_mask, 0] = np.maximum(0.78*blend[stable_mask,0] + 0.22*p12_stable, STABLE_FLOORS[12])
    blend[stable_mask, 1] = np.maximum(0.75*blend[stable_mask,1] + 0.25*p24_stable, STABLE_FLOORS[24])
    blend[stable_mask, 2] = np.maximum(0.80*blend[stable_mask,2] + 0.20*p48_stable, STABLE_FLOORS[48])

# 11c. Calibration pipeline — Submission A
oof_A  = enforce_monotonicity(oof_blend.copy())
oof_A  = right_tail_shift(oof_A)
oof_A  = enforce_monotonicity(oof_A)

test_A = enforce_monotonicity(test_blend.copy())
test_A = right_tail_shift(test_A)
test_A = enforce_monotonicity(test_A)

print('Submission A (baseline) calibration done.')

# 11d. Submission B — isotonic on far-zone + left-tail push
print('\nFitting isotonic on far-zone OOF...')
iso_24, gain_24 = fit_isotonic_far_zone(oof_A, time_values, event_values, far_train, 24, 1)
iso_48, gain_48 = fit_isotonic_far_zone(oof_A, time_values, event_values, far_train, 48, 2)

oof_B  = oof_A.copy()
test_B = test_A.copy()
oof_B[far_train, 1]  = iso_24.predict(oof_A[far_train, 1])
oof_B[far_train, 2]  = iso_48.predict(oof_A[far_train, 2])
test_B[far_test, 1]  = iso_24.predict(test_A[far_test, 1])
test_B[far_test, 2]  = iso_48.predict(test_A[far_test, 2])

oof_B  = left_tail_push(oof_B,  far_train)
test_B = left_tail_push(test_B, far_test)
oof_B  = enforce_monotonicity(oof_B)
test_B = enforce_monotonicity(test_B)

print('Submission B (isotonic) calibration done.')

In [ ]:
# ── Step 12: OOF Scoring ──────────────────────────────────────────────────
print('=' * 60)
print('OOF SCORES')
print('=' * 60)

for lbl, oof in [('A (baseline)', oof_A), ('B (isotonic)', oof_B)]:
    hybrid, c_idx, wb = compute_hybrid_score(
        time_values, event_values, oof[:,1], oof[:,2], oof[:,3])
    b12 = compute_brier(time_values, event_values, oof[:,0], 12)
    b24 = compute_brier(time_values, event_values, oof[:,1], 24)
    b48 = compute_brier(time_values, event_values, oof[:,2], 48)
    print(f'\nSubmission {lbl}')
    print(f'  Hybrid: {hybrid:.5f}  C-Index: {c_idx:.4f}  WBrier: {wb:.5f}')
    print(f'  B12={b12:.5f}  B24={b24:.5f}  B48={b48:.5f}')

    # Per-zone breakdown
    for zone_lbl, mask in [('near', near_train), ('far', far_train),
                             ('stable', stable_train), ('sparse', sparse_train)]:
        b24z = compute_brier(time_values[mask], event_values[mask], oof[mask,1], 24)
        b48z = compute_brier(time_values[mask], event_values[mask], oof[mask,2], 48)
        print(f'  {zone_lbl:8s}: B24={b24z:.5f}  B48={b48z:.5f}')

print(f'\nOOF delta B-A: {compute_hybrid_score(time_values, event_values, oof_B[:,1], oof_B[:,2], oof_B[:,3])[0] - compute_hybrid_score(time_values, event_values, oof_A[:,1], oof_A[:,2], oof_A[:,3])[0]:+.5f}')

In [ ]:
# ── Step 13: False Case Analysis ─────────────────────────────────────────
# Uses oof_A. Censored-unknown samples (event==0 AND time<horizon) are EXCLUDED.

print('=' * 70)
print('FALSE CASE ANALYSIS — OOF Predictions (oof_A, threshold=0.5)')
print('=' * 70)

THRESHOLD = 0.5
horizon_col_map = {12: 0, 24: 1, 48: 2, 72: 3}

# ── Global table ──────────────────────────────────────────────────────────
rows = []
for h, col in horizon_col_map.items():
    y_true, valid_mask = make_binary_target(time_values, event_values, h)
    valid_idx = np.where(valid_mask)[0]
    y_t = y_true[valid_idx]
    y_p = (oof_A[valid_idx, col] >= THRESHOLD).astype(int)
    prob_val = oof_A[valid_idx, col]

    tp = int(((y_p == 1) & (y_t == 1)).sum())
    tn = int(((y_p == 0) & (y_t == 0)).sum())
    fp = int(((y_p == 1) & (y_t == 0)).sum())
    fn = int(((y_p == 0) & (y_t == 1)).sum())
    fp_mean = float(prob_val[(y_p == 1) & (y_t == 0)].mean()) if fp > 0 else float('nan')
    fn_mean = float(prob_val[(y_p == 0) & (y_t == 1)].mean()) if fn > 0 else float('nan')

    rows.append({
        'horizon': f'{h}h', 'n_valid': int(valid_mask.sum()),
        'n_pos': int(y_t.sum()), 'n_neg': int((1 - y_t).sum()),
        'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn,
        'precision': round(tp / max(tp + fp, 1), 4),
        'recall':    round(tp / max(tp + fn, 1), 4),
        'FP_mean_prob': round(fp_mean, 4) if not np.isnan(fp_mean) else '-',
        'FN_mean_prob': round(fn_mean, 4) if not np.isnan(fn_mean) else '-',
    })

global_df = pd.DataFrame(rows).set_index('horizon')
print('\n── Global (all zones) ──')
print(global_df.to_string())

# ── Per-zone breakdown ────────────────────────────────────────────────────
print('\n── Per-zone FP / FN breakdown ──')
zone_rows = []
for zone_lbl, zone_mask in [
    ('near',   near_train),
    ('far',    far_train),
    ('stable', stable_train),
    ('sparse', sparse_train),
]:
    for h, col in horizon_col_map.items():
        y_true, valid_mask = make_binary_target(time_values, event_values, h)
        combined = valid_mask & zone_mask
        idx = np.where(combined)[0]
        if len(idx) == 0:
            continue
        y_t = y_true[idx]
        y_p = (oof_A[idx, col] >= THRESHOLD).astype(int)
        fp  = int(((y_p == 1) & (y_t == 0)).sum())
        fn  = int(((y_p == 0) & (y_t == 1)).sum())
        zone_rows.append({'zone': zone_lbl, 'horizon': f'{h}h',
                           'n': len(idx), 'pos': int(y_t.sum()),
                           'FP': fp, 'FN': fn})

zone_df = pd.DataFrame(zone_rows)
print(zone_df.pivot_table(index=['zone', 'n', 'pos'], columns='horizon',
                            values=['FP', 'FN'], aggfunc='first').to_string())

# ── Hardest false cases ───────────────────────────────────────────────────
print('\n── Hardest false cases (horizon=24h, high-confidence errors) ──')
y_true24, vm24 = make_binary_target(time_values, event_values, 24)
prob24 = oof_A[:, 1]
error_mask = vm24 & (np.abs(prob24 - y_true24) > 0.5)
hard_df = train_df[error_mask][['event_id', 'dist_min_ci_0_5h', 'low_temporal_resolution_0_5h',
                                   'time_to_hit_hours', 'event']].copy()
hard_df['pred_24h'] = prob24[error_mask].round(4)
hard_df['true_24h'] = y_true24[error_mask].astype(int)
hard_df['error_type'] = np.where(hard_df['pred_24h'] > 0.5, 'FP', 'FN')
hard_df['zone'] = np.where(
    hard_df['dist_min_ci_0_5h'] >= STRAT_THR, 'far',
    np.where(hard_df['low_temporal_resolution_0_5h'] == 1, 'sparse-near', 'stable-near')
)
print(hard_df.sort_values('pred_24h', ascending=False).to_string(index=False))
print(f'\nSummary: {error_mask.sum()} high-confidence errors at 24h')
print(hard_df['error_type'].value_counts().to_string())
print(hard_df.groupby(['zone', 'error_type']).size().rename('count').to_string())

In [ ]:
# ── Step 14: Submission Generation ───────────────────────────────────────

# Sanity checks
for lbl, test_final in [('A', test_A), ('B', test_B)]:
    sub = pd.DataFrame({
        'event_id': test_df['event_id'].values,
        'prob_12h':  test_final[:, 0],
        'prob_24h':  test_final[:, 1],
        'prob_48h':  test_final[:, 2],
        'prob_72h':  test_final[:, 3],
    })
    # Align to sample_submission event_id order
    sub = sample_sub[['event_id']].merge(sub, on='event_id', how='left')

    # Verify monotonicity
    mono_ok = (sub[['prob_12h','prob_24h','prob_48h','prob_72h']]
               .diff(axis=1).iloc[:, 1:] >= -1e-9).all().all()
    # Verify stable-near floors
    stable_test_ids = test_df.loc[stable_test, 'event_id'].values
    stable_sub = sub[sub['event_id'].isin(stable_test_ids)]
    floors_ok  = (
        (stable_sub['prob_12h'] >= STABLE_FLOORS[12] - 1e-6).all() and
        (stable_sub['prob_24h'] >= STABLE_FLOORS[24] - 1e-6).all() and
        (stable_sub['prob_48h'] >= STABLE_FLOORS[48] - 1e-6).all()
    )

    output_path = OUTPUT_PATH_A if lbl == 'A' else OUTPUT_PATH_B
    sub.to_csv(output_path, index=False)

    print(f'\n── Submission {lbl} ──')
    print(f'  Saved: {output_path}')
    print(f'  Monotonicity OK: {mono_ok}')
    print(f'  Stable-near floors OK: {floors_ok}')
    print(sub.describe().round(4).to_string())